# Image Classification VIT Base

https://colab.research.google.com/drive/1Oq47sBMjkZw3ta5nxmUKWp9S9R8RYK3j#scrollTo=J3zVvVc4Dcfg

In [1]:
from transformers import ViTForImageClassification, ViTFeatureExtractor, Trainer, TrainingArguments
from PIL import Image
import torch
import pandas as pd
import os
from torch.utils.data import Dataset, random_split
from google.colab import userdata

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# model_name = "./models/models--google--vit-base-patch16-224/snapshots/3f49326eb077187dfe1c2a2bb15fbd74e6ab91e3"
model_name = "google/vit-base-patch16-224"
model = ViTForImageClassification.from_pretrained(model_name, num_labels=2,ignore_mismatched_sizes=True)
feature_extractor = ViTFeatureExtractor.from_pretrained(model_name)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224 and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([1000]) in the checkpoint and torch.Size([2]) in the model instantiated
- classifier.weight: found shape torch.Size([1000, 768]) in the checkpoint and torch.Size([2, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/vit/feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [ ]:
# for param in model.parameters():
#     param.requires_grad = True

# for param in model.vit.encoder.layer[:-7].parameters():
#     param.requires_grad = True

## Transform Dataset

In [ ]:
class HouseDataset(Dataset):
    def __init__(self, df, image_folder, feature_extractor):
        self.df = df
        self.image_folder = image_folder
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        image_path = os.path.join(self.image_folder, self.df.iloc[idx]["image_name"])
        image = Image.open(image_path).convert("RGB")
        inputs = self.feature_extractor(images=image, return_tensors="pt")
        label = torch.tensor(self.df.iloc[idx]["class"], dtype=torch.long)
        return {"pixel_values": inputs["pixel_values"].squeeze(), "labels": label}

## Load Kaggle Dataset

In [ ]:
def download_kaggle(kaggle_command="kaggle competitions download -c liver-fibrosis-severity-prediction"):

  # Get Kaggle Key
  kaggle_username = userdata.get("KAGGLE_USER")
  kaggle_key = userdata.get("KAGGLE_KEY")
  if not kaggle_username or not kaggle_key:
      print("Error: Kaggle_USERNAME or Kaggle_KEY not found in Colab Secrets.")
      return

  # Write the credentials to ~/.kaggle/kaggle.json
  kaggle_dir = os.path.expanduser("~/.kaggle")
  os.makedirs(kaggle_dir, exist_ok=True)

  # Create JSON
  kaggle_json_path = os.path.join(kaggle_dir, "kaggle.json")
  with open(kaggle_json_path, "w") as f:
      f.write(f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}')
  os.chmod(kaggle_json_path, 0o600)

  try:
      os.system(kaggle_command)
      print("\n--- Download complete! ---")
      os.system("ls -la")
      os.system("unzip -o '*.zip' && rm -f *.zip")
      os.system("ls -la")

  except Exception as e:
      print(f"An error occurred during download: {e}")

In [ ]:
PATH = ""
download_kaggle(kaggle_command=PATH)

## Load Dataframe

In [4]:
df = pd.read_csv("./data/train.csv")
image_folder = "./data/train/train"

# Transform Dataset
dataset = HouseDataset(df, image_folder, feature_extractor)

# Split Train and Test Dataset
train_dataset, eval_dataset = random_split(dataset, [0.8, 0.2])

FileNotFoundError: [Errno 2] No such file or directory: './data/train.csv'

## Setup Metrics Score

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    # If predictions are logits, take argmax to get class labels
    predictions = predictions.argmax(-1)
    accuracy = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average="weighted")
    return {
        "accuracy": accuracy,
        "f1": f1,
    }

## Training Model

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,
    num_train_epochs=25,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none",
    fp16 = True,
    dataloader_num_workers = 4
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,  # Add evaluation dataset
    compute_metrics = compute_metrics
)

In [ ]:
trainer.train()

## Inferences

In [ ]:
def predict(images_folder, model, feature_extractor, device):
    model.eval()
    predictions = []
    image_files = [f for f in os.listdir(images_folder) if f.endswith(".jpg") or f.endswith(".png")]

    for image_file in image_files:
        image_path = os.path.join(images_folder, image_file)
        image = Image.open(image_path).convert("RGB")
        inputs = feature_extractor(images=image, return_tensors="pt").to(device)
        with torch.no_grad():
            outputs = model(**inputs)
            predicted_label = torch.argmax(outputs.logits, dim=-1).item()
        predictions.append((image_file, predicted_label))

    return predictions

In [ ]:
test_folder = "./data/test/test"
predictions = predict(test_folder, model, feature_extractor,device)

In [ ]:
submission = pd.read_csv("./data/sample_submission.csv")
submission

In [ ]:
submission = pd.DataFrame(predictions)
submission

In [ ]:
submission = submission.rename(columns={0: "id", 1: "answer"})
submission["id"] = submission["id"].str.replace(".jpg", "")

In [ ]:
submission.to_csv("./submission/ViTsubmission.csv",index=False)